# Notebook 1: Data Collection
## Songwriter Style Analysis - Enhanced Data Collection

**Objective**: Collect 50+ songs per songwriter for improved accuracy

**Target Songwriters**: 12 professional songwriters/producers

**APIs Used**:
- Genius API: Lyrics and songwriter credits
- Last.fm API: Popularity metrics and tags

**Expected Output**: songs_data_final.csv with 600+ songs

## Step 1: Import Libraries and Setup

In [ ]:
# Import essential libraries
import lyricsgenius as lg
import pylast
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Step 2: Configure API Credentials

In [ ]:
# Import API credentials
from config import GENIUS_TOKEN, LASTFM_API_KEY, LASTFM_API_SECRET

# Initialize Genius API
genius = lg.Genius(
    GENIUS_TOKEN,
    skip_non_songs=True,
    remove_section_headers=True,
    verbose=False,
    timeout=15,
    retries=3
)

# Initialize Last.fm API
lastfm_network = pylast.LastFMNetwork(
    api_key=LASTFM_API_KEY,
    api_secret=LASTFM_API_SECRET
)

print("API clients initialized successfully")
print(f"Genius Token: {GENIUS_TOKEN[:20]}...")
print(f"Last.fm API Key: {LASTFM_API_KEY[:20]}...")

## Step 3: Test API Connections

In [ ]:
# Test Genius API
print("Testing Genius API connection...")
print("="*60)

try:
    test_song = genius.search_song("Blinding Lights", "The Weeknd")
    if test_song:
        print("[SUCCESS] Genius API connection working")
        print(f"Song: {test_song.title}")
        print(f"Artist: {test_song.artist}")
        print(f"Lyrics length: {len(test_song.lyrics)} characters")
        
        # Check for writer credits
        if hasattr(test_song, '_body') and 'writer_artists' in test_song._body:
            writers = [w['name'] for w in test_song._body['writer_artists']]
            print(f"Writers: {', '.join(writers)}")
    else:
        print("[ERROR] Song not found")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)

# Test Last.fm API
print("Testing Last.fm API connection...")
print("="*60)

try:
    track = lastfm_network.get_track("The Weeknd", "Blinding Lights")
    print("[SUCCESS] Last.fm API connection working")
    print(f"Track: {track.get_name()}")
    print(f"Artist: {track.get_artist()}")
    print(f"Playcount: {track.get_playcount():,}")
    print(f"Listeners: {track.get_listener_count():,}")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)
print("API connection tests complete")

## Step 4: Define Target Songwriters (Expanded List)

In [ ]:
# Enhanced songwriter list with more artists for better data collection
target_writers = {
    'Max Martin': {
        'known_for': 'Pop hits, catchy hooks',
        'notable_artists': ['Taylor Swift', 'The Weeknd', 'Ariana Grande', 'Katy Perry', 
                           'Maroon 5', 'P!nk', 'Backstreet Boys', 'Britney Spears']
    },
    'Sia Furler': {
        'known_for': 'Powerful vocals, emotional lyrics',
        'notable_artists': ['Rihanna', 'Beyoncé', 'David Guetta', 'Sia', 
                           'Flo Rida', 'Eminem', 'Christina Aguilera']
    },
    'Ryan Tedder': {
        'known_for': 'Anthemic choruses, piano-driven',
        'notable_artists': ['Beyoncé', 'Adele', 'Taylor Swift', 'OneRepublic', 
                           'Leona Lewis', 'Ed Sheeran', 'Jonas Brothers']
    },
    'Julia Michaels': {
        'known_for': 'Personal, confessional lyrics',
        'notable_artists': ['Selena Gomez', 'Justin Bieber', 'Demi Lovato', 
                           'Hailee Steinfeld', 'Linkin Park', 'Gwen Stefani']
    },
    'The-Dream': {
        'known_for': 'R&B melodies, romantic themes',
        'notable_artists': ['Beyoncé', 'Rihanna', 'Mariah Carey', 'Justin Bieber', 
                           'Jay-Z', 'Kanye West']
    },
    'Jack Antonoff': {
        'known_for': '80s-inspired production, indie-pop',
        'notable_artists': ['Taylor Swift', 'Lorde', 'Lana Del Rey', 'Bleachers', 
                           'St. Vincent', 'Carly Rae Jepsen', 'The Chicks']
    },
    'Shellback': {
        'known_for': 'Pop production, Max Martin collaborator',
        'notable_artists': ['Taylor Swift', 'P!nk', 'Maroon 5', 'Ariana Grande', 
                           'Britney Spears', 'Usher', 'Pitbull']
    },
    'Benny Blanco': {
        'known_for': 'Pop and hip-hop fusion',
        'notable_artists': ['Ed Sheeran', 'Justin Bieber', 'Halsey', 'Maroon 5', 
                           'Kesha', 'Rihanna', 'Katy Perry']
    },
    'Stargate': {
        'known_for': 'R&B and pop production',
        'notable_artists': ['Rihanna', 'Beyoncé', 'Ne-Yo', 'Katy Perry', 
                           'Sam Smith', 'Wiz Khalifa', 'Coldplay']
    },
    'Dr. Luke': {
        'known_for': 'Pop hits, electronic production',
        'notable_artists': ['Katy Perry', 'Kesha', 'Doja Cat', 'Kim Petras', 
                           'P!nk', 'Miley Cyrus', 'Avril Lavigne']
    },
    'Greg Kurstin': {
        'known_for': 'Alternative pop, diverse production',
        'notable_artists': ['Adele', 'Beck', 'Foo Fighters', 'Sia', 
                           'P!nk', 'Kelly Clarkson', 'Halsey']
    },
    'Diplo': {
        'known_for': 'Electronic, dancehall, pop fusion',
        'notable_artists': ['Major Lazer', 'Justin Bieber', 'M.I.A.', 'Beyoncé', 
                           'Skrillex', 'Madonna', 'Usher']
    }
}

print("Target Songwriters Configuration")
print("="*60)
for writer, info in target_writers.items():
    print(f"\n{writer}")
    print(f"  Style: {info['known_for']}")
    print(f"  Artists ({len(info['notable_artists'])}): {', '.join(info['notable_artists'][:4])}...")

print("\n" + "="*60)
print(f"Total Songwriters: {len(target_writers)}")
print(f"Target: 50+ songs per songwriter")
print(f"Expected Total: 600+ songs")

## Step 5: Helper Functions for Data Collection

In [ ]:
def extract_song_data(song, target_songwriter):
    """
    Extract comprehensive data from a Genius song object
    
    Args:
        song: Genius song object
        target_songwriter: Name of the songwriter we're collecting for
    
    Returns:
        dict: Song data or None if invalid
    """
    if not song or not hasattr(song, 'lyrics'):
        return None
    
    # Skip songs without lyrics or very short lyrics
    if not song.lyrics or len(song.lyrics.strip()) < 100:
        return None
    
    song_data = {
        'title': song.title,
        'artist': song.artist,
        'lyrics': song.lyrics,
        'target_songwriter': target_songwriter,
        'writers': 'Unknown',
        'producers': 'Unknown',
        'release_date': None,
        'url': song.url if hasattr(song, 'url') else None,
        'pageviews': 0,
        'lastfm_playcount': 0,
        'lastfm_listeners': 0,
        'lastfm_tags': ''
    }
    
    # Extract metadata from Genius
    if hasattr(song, '_body'):
        metadata = song._body
        
        # Get writers
        if 'writer_artists' in metadata:
            writers = [w['name'] for w in metadata['writer_artists']]
            song_data['writers'] = ', '.join(writers)
        
        # Get producers
        if 'producer_artists' in metadata:
            producers = [p['name'] for p in metadata['producer_artists']]
            song_data['producers'] = ', '.join(producers)
        
        # Get release date
        if 'release_date' in metadata and metadata['release_date']:
            song_data['release_date'] = metadata['release_date']
        
        # Get pageviews
        if 'stats' in metadata and 'pageviews' in metadata['stats']:
            song_data['pageviews'] = metadata['stats']['pageviews']
    
    return song_data


def get_lastfm_data(artist_name, song_title):
    """
    Get popularity metrics from Last.fm
    
    Args:
        artist_name: Artist name
        song_title: Song title
    
    Returns:
        dict: Last.fm metrics
    """
    lastfm_data = {
        'playcount': 0,
        'listeners': 0,
        'tags': ''
    }
    
    try:
        track = lastfm_network.get_track(artist_name, song_title)
        lastfm_data['playcount'] = track.get_playcount()
        lastfm_data['listeners'] = track.get_listener_count()
        
        # Get tags
        tags = track.get_top_tags(limit=3)
        lastfm_data['tags'] = ', '.join([tag.item.get_name() for tag in tags])
    except:
        pass
    
    return lastfm_data


def save_checkpoint(songwriter_name, songs_data):
    """
    Save checkpoint for a songwriter's collected data
    
    Args:
        songwriter_name: Name of songwriter
        songs_data: List of song dictionaries
    """
    if not os.path.exists('data'):
        os.makedirs('data')
    
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    df = pd.DataFrame(songs_data)
    df.to_csv(checkpoint_file, index=False)
    print(f"  [CHECKPOINT] Saved {len(songs_data)} songs to {checkpoint_file}")


print("Helper functions defined successfully")
print("  - extract_song_data(): Extract metadata from Genius")
print("  - get_lastfm_data(): Get popularity metrics")
print("  - save_checkpoint(): Save progress during collection")

## Step 6: Enhanced Data Collection (50+ Songs Per Songwriter)

In [ ]:
# Enhanced data collection with improved strategies
print("STARTING ENHANCED DATA COLLECTION")
print("="*60)
print(f"Target: {len(target_writers)} songwriters")
print(f"Goal: 50+ songs per songwriter")
print(f"Expected total: 600+ songs")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Storage
all_songs_data = []
songwriter_stats = {}

# Rate limiting configuration
DELAY_BETWEEN_SONGS = 1.5  # seconds
DELAY_BETWEEN_ARTISTS = 3.0  # seconds
DELAY_AFTER_RATE_LIMIT = 60  # seconds

for songwriter, info in target_writers.items():
    print(f"\n{'='*60}")
    print(f"COLLECTING: {songwriter}")
    print(f"Style: {info['known_for']}")
    print(f"{'='*60}")
    
    songwriter_songs = []
    target_count = 50  # Increased from 25 to 50
    collected_titles = set()  # Prevent duplicates
    
    # Strategy 1: Collect from multiple artists
    for artist_name in info['notable_artists']:
        if len(songwriter_songs) >= target_count:
            break
        
        print(f"\n  Searching {artist_name} songs...")
        
        try:
            # Search for artist on Genius
            artist = genius.search_artist(artist_name, max_songs=15)
            
            if artist and hasattr(artist, 'songs'):
                print(f"    Found {len(artist.songs)} songs")
                
                for song in artist.songs:
                    if len(songwriter_songs) >= target_count:
                        break
                    
                    # Skip duplicates
                    song_key = f"{song.title.lower()}_{song.artist.lower()}"
                    if song_key in collected_titles:
                        continue
                    
                    # Extract song data
                    song_data = extract_song_data(song, songwriter)
                    
                    if song_data:
                        # Check if target songwriter is credited
                        writers_lower = song_data['writers'].lower()
                        producers_lower = song_data['producers'].lower()
                        songwriter_lower = songwriter.lower()
                        
                        # More flexible matching
                        is_credited = (
                            songwriter_lower in writers_lower or
                            songwriter_lower in producers_lower or
                            any(part in writers_lower for part in songwriter_lower.split()) or
                            any(part in producers_lower for part in songwriter_lower.split())
                        )
                        
                        if is_credited:
                            # Get Last.fm data
                            lastfm_data = get_lastfm_data(song_data['artist'], song_data['title'])
                            song_data['lastfm_playcount'] = lastfm_data['playcount']
                            song_data['lastfm_listeners'] = lastfm_data['listeners']
                            song_data['lastfm_tags'] = lastfm_data['tags']
                            
                            songwriter_songs.append(song_data)
                            collected_titles.add(song_key)
                            
                            print(f"    [OK] {song_data['title']} - {len(songwriter_songs)}/{target_count}")
                            
                            time.sleep(DELAY_BETWEEN_SONGS)
            
            time.sleep(DELAY_BETWEEN_ARTISTS)
            
        except Exception as e:
            print(f"    [ERROR] {str(e)}")
            if 'rate limit' in str(e).lower():
                print(f"    [WAIT] Rate limit hit, waiting {DELAY_AFTER_RATE_LIMIT}s...")
                time.sleep(DELAY_AFTER_RATE_LIMIT)
            continue
    
    # Save checkpoint
    if songwriter_songs:
        save_checkpoint(songwriter, songwriter_songs)
        all_songs_data.extend(songwriter_songs)
        songwriter_stats[songwriter] = len(songwriter_songs)
    
    print(f"\n  COLLECTED: {len(songwriter_songs)} songs for {songwriter}")
    print(f"  PROGRESS: {len(all_songs_data)} total songs collected")

print("\n" + "="*60)
print("DATA COLLECTION COMPLETE")
print("="*60)
print(f"\nCollection Summary:")
for songwriter, count in songwriter_stats.items():
    print(f"  {songwriter}: {count} songs")
print(f"\nTotal Songs Collected: {len(all_songs_data)}")
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 7: Save Final Dataset

In [ ]:
# Create DataFrame from collected data
df_songs = pd.DataFrame(all_songs_data)

print("Dataset Overview")
print("="*60)
print(f"Total Songs: {len(df_songs)}")
print(f"Total Columns: {len(df_songs.columns)}")
print(f"\nColumns: {list(df_songs.columns)}")

print("\nSongs per Songwriter:")
print(df_songs['target_songwriter'].value_counts())

print("\nDataset Statistics:")
print(f"  Average lyrics length: {df_songs['lyrics'].str.len().mean():.0f} characters")
print(f"  Median lyrics length: {df_songs['lyrics'].str.len().median():.0f} characters")
print(f"  Missing values: {df_songs.isnull().sum().sum()}")

# Save to CSV
output_file = 'data/songs_data_final.csv'
df_songs.to_csv(output_file, index=False)
print(f"\n[SAVED] Dataset saved to {output_file}")

# Also save as JSON backup
json_file = 'data/songs_data_final.json'
df_songs.to_json(json_file, orient='records', indent=2)
print(f"[SAVED] JSON backup saved to {json_file}")

# Save metadata
metadata = {
    'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_songs': len(df_songs),
    'songwriters': list(songwriter_stats.keys()),
    'songs_per_songwriter': songwriter_stats
}

import json
metadata_file = 'data/collection_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"[SAVED] Metadata saved to {metadata_file}")

print("\n" + "="*60)
print("DATA COLLECTION NOTEBOOK COMPLETE")
print("="*60)
print(f"\nNext Step: Run 02_preprocessing.ipynb")

In [ ]:
# Display sample of collected data
print("Sample of Collected Data:")
print("="*60)
df_songs[['title', 'artist', 'target_songwriter', 'writers', 'lastfm_playcount']].head(10)